# Conexiones residuales

**Capítulo 4 · Universidad de las Hespérides**

Adaptación al español de *Dive into Deep Learning*, Aston Zhang, Zachary C. Lipton, Mu Li y Alexander J. Smola.
Fuente: `locked/chapter_convolutional-modern/resnet.ipynb` · [Lección original](https://d2l.ai/chapter_convolutional-modern/resnet.html).
Texto adaptado bajo [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/). [Procedencia y cambios](../PROCEDENCIA.md).
Se conserva la secuencia de las celdas y de los ejercicios; las notas de Hespérides se identifican expresamente.

**Entorno:** ejecuta `uv sync` en la raíz y selecciona su Python como kernel. Las descargas se realizan una vez y quedan en `data/`.
Por defecto, el soporte limita los entrenamientos de `Trainer` a tres épocas y 1024/256 ejemplos para CPU.
Para repetir el régimen completo, inicia Jupyter con `HESPERIDES_COMPLETO=1`. Los ejemplos visuales pequeños conservan su propia configuración explícita.
Los datos de texto en inglés o francés son entradas de los experimentos originales y mantienen su idioma.


In [ ]:
from pathlib import Path
import sys
RAIZ = Path.cwd() if (Path.cwd() / "laboratorio").exists() else Path.cwd().parent
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
from laboratorio import d2l, configurar, epocas
configurar()


# Redes residuales (ResNet) y ResNeXt
<a id="sec_resnet"></a>

A medida que diseñamos redes cada vez más profundas se hace imperativo entender cómo agregar capas puede aumentar la complejidad y expresividad de la red. Aún más importante es la capacidad de diseñar redes donde agregar capas hace que las redes sean estrictamente más expresivas que sólo diferentes. Para hacer algún progreso necesitamos un poco de matemáticas.


In [ ]:
import torch
from torch import nn
from torch.nn import functional as F
from laboratorio import d2l

## Clases de funciones
Considere $\mathcal{F}$, la clase de funciones que una arquitectura de red específica (junto con las tasas de aprendizaje y otras configuraciones de hiperparametros) puede alcanzar. Es decir, para todo $f \in \mathcal{F}$ existe algún conjunto de parámetros (por ejemplo, pesos y sesgos) que se pueden obtener mediante el entrenamiento en un conjunto de datos adecuado. Supongamos que $f^*$ es la función de "verdad" que realmente nos gustaría encontrar. Si está en $\mathcal{F}$, estamos en buena forma, pero típicamente no vamos a ser muy afortunados. En su lugar, vamos a tratar de encontrar algunos $f^*_\mathcal{F}$ que es nuestra mejor apuesta dentro de $\mathcal{F}$. Por ejemplo, dado un conjunto de datos con características $\mathbf{X}$ y etiquetas $\mathbf{y}$, podríamos intentar encontrarlo resolviendo el siguiente problema de optimización:

$$f^*_\mathcal{F} \stackrel{\textrm{def}}{=} \mathop{\mathrm{argmin}}_f L(\mathbf{X}, \mathbf{y}, f) \textrm{ subject to } f \in \mathcal{F}.$$

Sabemos que la regularización [tikhonov1977solutions,morozov2012methods](https://d2l.ai/chapter_references/zreferences.html) puede controlar la complejidad de $\mathcal{F}$ y lograr consistencia, por lo que un tamaño mayor de los datos de entrenamiento generalmente conduce a una mejor $f^*_\mathcal{F}$. Es razonable suponer que si diseñamos una arquitectura diferente y más potente $\mathcal{F}'$ deberíamos llegar a un mejor resultado. En otras palabras, esperaríamos que $f^*_{\mathcal{F}'}$ es "mejor" que $f^*_{\mathcal{F}}$. Sin embargo, si $\mathcal{F} \not\subseteq \mathcal{F}'$ no hay garantía de que esto incluso debería suceder. De hecho, $f^*_{\mathcal{F}'}$ bien podría ser peor. Como ilustra [Referencia fig_functionclasses](https://d2l.ai/chapter_convolutional-modern/resnet.html#fig-functionclasses), para las clases de funciones no anidadas, una clase de funciones más grande no siempre se acerca a la función de "verdad" $f^*$. Por ejemplo, a la izquierda de [Referencia fig_functionclasses](https://d2l.ai/chapter_convolutional-modern/resnet.html#fig-functionclasses), aunque $\mathcal{F}_3$ está más cerca de $f^*$ que $\mathcal{F}_1$, $\mathcal{F}_6$ se aleja y no hay garantía de que el aumento de la complejidad pueda reducir la distancia de $f^*$. Con las clases de función anidada donde $\mathcal{F}_1 \subseteq \cdots \subseteq \mathcal{F}_6$ a la derecha de [Referencia fig_functionclasses](https://d2l.ai/chapter_convolutional-modern/resnet.html#fig-functionclasses), podemos evitar el problema mencionado de las clases de funciones no anidadas.

![En clases de funciones no anidadas, ampliar el área no garantiza acercarse a la función objetivo $\mathit{f}^*$. Las clases anidadas evitan este problema.](../recursos/originales/functionclasses.svg)
<a id="fig_functionclasses"></a>

Así, sólo si las clases de funciones más grandes contienen las más pequeñas se garantiza que aumentarlas aumenta estrictamente el poder expresivo de la red. Para las redes neuronales profundas, si podemos entrenar la capa recién añadida en una función de identidad $f(\mathbf{x}) = \mathbf{x}$, el nuevo modelo será tan eficaz como el modelo original. Como el nuevo modelo puede obtener una mejor solución para adaptarse al conjunto de datos de entrenamiento, la capa añadida podría hacer más fácil reducir los errores de entrenamiento.

Esta es la pregunta que [He.Zhang.Ren.ea.2016](https://d2l.ai/chapter_references/zreferences.html) consideró al trabajar en modelos de visión computarizada muy profundos. En el corazón de su propuesta *red residual* (*ResNet*) es la idea de que cada capa adicional debería contener más fácilmente la función de identidad como uno de sus elementos. Estas consideraciones son bastante profundas pero llevaron a una solución sorprendentemente simple, un *bloque residual*. Con ella, ResNet ganó el desafío de reconocimiento visual a gran escala de ImageNet en 2015. El diseño tuvo una profunda influencia en cómo construir redes neuronales profundas. Por ejemplo, bloques residuales se han añadido a redes recurrentes [prakash2016neural,kim2017residual](https://d2l.ai/chapter_references/zreferences.html). Del mismo modo, los transformadores [Vaswani.Shazeer.Parmar.ea.2017](https://d2l.ai/chapter_references/zreferences.html) los utilizan para apilar muchas capas de redes de manera eficiente. También se utiliza en redes neuronales gráficas [Kipf.Welling.2016](https://d2l.ai/chapter_references/zreferences.html) y, como concepto básico, se ha utilizado ampliamente en la visión computacional [Redmon.Farhadi.2018,Ren.He.Girshick.ea.2015](https://d2l.ai/chapter_references/zreferences.html). Tenga en cuenta que las redes residuales están predadas por redes de carreteras [srivastava2015highway](https://d2l.ai/chapter_references/zreferences.html) que comparten parte de la motivación, aunque sin la elegante parametrización en torno a la función de identidad.

## Bloques residuales
<a id="subsec_residual-blks"></a>

Centrémonos en una parte local de una red neural, como se muestra en [Referencia fig_residual_block](https://d2l.ai/chapter_convolutional-modern/resnet.html#fig-residual-block). Denotemos la entrada de $\mathbf{x}$. Asumimos que $f(\mathbf{x})$, el mapeo subyacente deseado que queremos obtener mediante el aprendizaje, se utilizará como entrada a la función de activación en la parte superior. A la izquierda, la porción dentro del mapeo de línea punteada debe aprender directamente $f(\mathbf{x})$. A la derecha, la porción dentro del mapeo de línea punteada necesita aprender el mapeo *residual* $g(\mathbf{x}) = f(\mathbf{x}) - \mathbf{x}$, que es cómo el bloque residual deriva su nombre. Si el mapeo de identidad $f(\mathbf{x}) = \mathbf{x}$ es el mapeo subyacente deseado, el mapeo residual asciende a $g(\mathbf{x}) = 0$ y por lo tanto es más fácil de aprender: solo necesitamos empujar los pesos y sesgos de la capa de peso superior (por ejemplo, capa totalmente conectada y capa convolucional) dentro del mapeo de línea punteada a cero. La figura derecha ilustra el bloque *residual* de ResNet, donde la línea sólida que lleva la entrada de capa $\mathbf{x}$ al operador se llama una conexión adicional* (o *acortado*).

![El bloque convencional (izquierda) aprende directamente $\mathit{f}(\mathbf{x})$. El residual (derecha) aprende $\mathit{g}(\mathbf{x})=\mathit{f}(\mathbf{x})-\mathbf{x}$, facilitando representar la identidad.](../recursos/originales/residual-block.svg)
<a id="fig_residual_block"></a>

ResNet tiene el diseño completo de capa convolucional $3\times 3$ de VGG. El bloque residual tiene dos capas convolucionales $3\times 3$ con el mismo número de canales de salida. Cada capa convolucional es seguida por una capa de normalización por lotes (BatchNorm) y una función de activación ReLU. Entonces, omitimos estas dos operaciones de convolución y añadimos la entrada directamente antes de la función de activación ReLU final. Este tipo de diseño requiere que la salida de las dos capas convolucionales tenga la misma forma que la entrada, para que puedan ser agregadas juntas. Si queremos cambiar el número de canales, necesitamos introducir una capa convolucional $1\times 1$ adicional para transformar la entrada en la forma deseada para la operación de adición. Echemos un vistazo al código de abajo.


In [ ]:
class Residual(nn.Module):  #@save
    """El bloque residual de los modelos ResNet."""
    def __init__(self, num_channels, use_1x1conv=False, strides=1):
        super().__init__()
        self.conv1 = nn.LazyConv2d(num_channels, kernel_size=3, padding=1,
                                   stride=strides)
        self.conv2 = nn.LazyConv2d(num_channels, kernel_size=3, padding=1)
        if use_1x1conv:
            self.conv3 = nn.LazyConv2d(num_channels, kernel_size=1,
                                       stride=strides)
        else:
            self.conv3 = None
        self.bn1 = nn.LazyBatchNorm2d()
        self.bn2 = nn.LazyBatchNorm2d()

    def forward(self, X):
        Y = F.relu(self.bn1(self.conv1(X)))
        Y = self.bn2(self.conv2(Y))
        if self.conv3:
            X = self.conv3(X)
        Y += X
        return F.relu(Y)

Este código genera dos tipos de redes: una en la que añadimos la entrada a la salida antes de aplicar la no linealidad ReLU cuando `use_1x1conv=False`; y otra en la que ajustamos los canales y la resolución mediante una convolución $1 \times 1$ antes de añadir. [Referencia fig_resnet_block](https://d2l.ai/chapter_convolutional-modern/resnet.html#fig-resnet-block) ilustra esto.

![Bloque ResNet con y sin convolución $1 \times 1$, que adapta la forma de la entrada para poder sumarla.](../recursos/originales/resnet-block.svg)
<a id="fig_resnet_block"></a>

Ahora veamos ** una situación donde la entrada y la salida son de la misma forma**, donde $1 \times 1$ convolution no es necesario.


In [ ]:
blk = Residual(3)
X = torch.randn(4, 3, 6, 6)
blk(X).shape

También tenemos la opción de ** reducir a la mitad la altura y el ancho de salida mientras aumenta el número de canales de salida**. En este caso utilizamos $1 \times 1$ convolutions via `use_1x1conv=True`. Esto es útil al principio de cada bloque de ResNet para reducir la dimensión espacial a través de `strides=2`.


In [ ]:
blk = Residual(6, use_1x1conv=True, strides=2)
blk(X).shape

## Modelo ResNet

Las dos primeras capas de ResNet son las mismas que las de la GoogLeNet que describimos antes: la capa convolucional $7\times 7$ con 64 canales de salida y un paso de 2 es seguida por la capa de max-pooling $3\times 3$ con un paso de 2. La diferencia es la capa de normalización por lotes (BatchNorm) añadida después de cada capa convolucional en ResNet.


In [ ]:
class ResNet(d2l.Classifier):
    def b1(self):
        return nn.Sequential(
            nn.LazyConv2d(64, kernel_size=7, stride=2, padding=3),
            nn.LazyBatchNorm2d(), nn.ReLU(),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1))

GoogLeNet utiliza cuatro módulos formados por bloques de Inception. Sin embargo, ResNet utiliza cuatro módulos compuestos por bloques residuales, cada uno de los cuales utiliza varios bloques residuales con el mismo número de canales de salida. El número de canales en el primer módulo es el mismo que el número de canales de entrada. Como ya se ha utilizado una capa de max-pooling con un paso de 2 no es necesario reducir la altura y el ancho. En el primer bloque residual para cada uno de los módulos posteriores, el número de canales se duplica en comparación con el del módulo anterior, y la altura y el ancho se reducen a la mitad.


In [ ]:
@d2l.add_to_class(ResNet)
def block(self, num_residuals, num_channels, first_block=False):
    blk = []
    for i in range(num_residuals):
        if i == 0 and not first_block:
            blk.append(Residual(num_channels, use_1x1conv=True, strides=2))
        else:
            blk.append(Residual(num_channels))
    return nn.Sequential(*blk)

### Nota docente de Hespérides

Un bloque residual calcula $y=x+F(x)$ cuando las formas coinciden; si cambian canales o resolución, utiliza una proyección para compatibilizarlas. La ruta identidad facilita el transporte de señales y gradientes, pero no hace desaparecer todos los problemas de optimización. Contrasta el número de parámetros y las formas con LeNet. VGG y EfficientNet amplían la historia arquitectónica de los apuntes; esta práctica prioriza el mecanismo residual frente a entrenar un catálogo completo.

Vínculo con los apuntes: sesión 4, «Conexiones residuales».


A continuación, agregamos todos los módulos a ResNet. Aquí, dos bloques residuales se utilizan para cada módulo. Por último, al igual que GoogLeNet, añadimos una capa de agrupación media global, seguido por la salida de capa totalmente conectada.


In [ ]:
@d2l.add_to_class(ResNet)
def __init__(self, arch, lr=0.1, num_classes=10):
    super(ResNet, self).__init__()
    self.save_hyperparameters()
    self.net = nn.Sequential(self.b1())
    for i, b in enumerate(arch):
        self.net.add_module(f'b{i+2}', self.block(*b, first_block=(i==0)))
    self.net.add_module('last', nn.Sequential(
        nn.AdaptiveAvgPool2d((1, 1)), nn.Flatten(),
        nn.LazyLinear(num_classes)))
    self.net.apply(d2l.init_cnn)

Hay cuatro capas convolucionales en cada módulo (excluyendo la capa convolucional $1\times 1$). Junto con la primera capa convolucional $7\times 7$ y la capa final totalmente conectada, hay 18 capas en total. Por lo tanto, este modelo se conoce comúnmente como ResNet-18. Configurando diferentes números de canales y bloques residuales en el módulo, podemos crear diferentes modelos de ResNet, como el más profundo de 152 capas ResNet-152. Aunque la arquitectura principal de ResNet es similar a la de GoogLeNet, la estructura de ResNet es más simple y fácil de modificar. Todos estos factores han resultado en el uso rápido y generalizado de ResNet. [Referencia fig_resnet18](https://d2l.ai/chapter_convolutional-modern/resnet.html#fig-resnet18) representa la totalidad de ResNet-18.

![Arquitectura ResNet-18.](../recursos/originales/resnet18-90.svg)
<a id="fig_resnet18"></a>

Antes de entrenar a ResNet, observemos cómo cambia la forma de entrada en diferentes módulos de ResNet**. Como en todas las arquitecturas anteriores, la resolución disminuye mientras que el número de canales aumenta hasta el punto en que una capa media de pooling global agrega todas las características.


In [ ]:
class ResNet18(ResNet):
    def __init__(self, lr=0.1, num_classes=10):
        super().__init__(((2, 64), (2, 128), (2, 256), (2, 512)),
                       lr, num_classes)

In [ ]:
ResNet18().layer_summary((1, 1, 96, 96))

## Entrenamiento

Entrenamos a ResNet en el conjunto de datos de Fashion-MNIST, al igual que antes. ResNet es una arquitectura muy potente y flexible. La trama que captura la pérdida de entrenamiento y validación ilustra una brecha significativa entre ambos gráficos, con la pérdida de entrenamiento siendo considerablemente menor. Para una red de esta flexibilidad, más datos de entrenamiento ofrecerían un beneficio distintivo para cerrar la brecha y mejorar la precisión.


In [ ]:
model = ResNet18(lr=0.01)
trainer = d2l.Trainer(max_epochs=10, num_gpus=1)
data = d2l.FashionMNIST(batch_size=128, resize=(96, 96))
model.apply_init([next(iter(data.get_dataloader(True)))[0]], d2l.init_cnn)
trainer.fit(model, data)

## ResNeXt
<a id="subsec_resnext"></a>

Uno de los desafíos que uno encuentra en el diseño de ResNet es el intercambio entre la no linealidad y la dimensionalidad dentro de un bloque dado. Es decir, podríamos añadir más no linealidad aumentando el número de capas, o aumentando el ancho de las convoluciones. Una estrategia alternativa es aumentar el número de canales que pueden transportar información entre bloques. Desafortunadamente, este último viene con una penalización cuadrática ya que el costo computacional de ingerir canales $c_\textrm{i}$ y emitir canales $c_\textrm{o}$ es proporcional a $\mathcal{O}(c_\textrm{i} \cdot c_\textrm{o})$ (ver nuestra discusión en [Referencia sec_channels](https://d2l.ai/chapter_convolutional-neural-networks/channels.html#sec-channels)).

Podemos inspirarnos en el bloque de Inception de [Referencia fig_inception](https://d2l.ai/chapter_convolutional-modern/googlenet.html#fig-inception) que tiene información que fluye a través del bloque en grupos separados. Aplicando la idea de múltiples grupos independientes al bloque de ResNet de [Referencia fig_resnet_block](https://d2l.ai/chapter_convolutional-modern/resnet.html#fig-resnet-block) llevó al diseño de ResNeXt [Xie.Girshick.Dollar.ea.2017](https://d2l.ai/chapter_references/zreferences.html). Diferente del smorgasbord de transformaciones en Inception, ResNeXt adopta la transformación *same* en todas las ramas, minimizando así la necesidad de ajuste manual de cada rama.

![Bloque ResNeXt. Con $\mathit{g}$ grupos, la convolución reduce por ese factor el coste aritmético frente a la densa; la aceleración real depende del hardware. Hay un cuello de botella cuando $\mathit{b}<\mathit{c}$.](../recursos/originales/resnext-block.svg)
<a id="fig_resnext_block"></a>

Romper una convolución de canales $c_\textrm{i}$ a $c_\textrm{o}$ en uno de los grupos $g$ del tamaño $c_\textrm{i}/g$ que genera salidas $g$ del tamaño $c_\textrm{o}/g$ se llama, bastante apropiadamente, una *convolución agrupada*. El costo computacional (proporcionalmente) se reduce de $\mathcal{O}(c_\textrm{i} \cdot c_\textrm{o})$ a $\mathcal{O}(g \cdot (c_\textrm{i}/g) \cdot (c_\textrm{o}/g)) = \mathcal{O}(c_\textrm{i} \cdot c_\textrm{o} / g)$, es decir, es $g$ veces más rápido. Aún mejor, el número de parámetros necesarios para generar la salida también se reduce de una matriz $c_\textrm{i} \times c_\textrm{o}$ a $g$ matrices más pequeñas de tamaño $(c_\textrm{i}/g) \times (c_\textrm{o}/g)$, de nuevo una reducción $g$ veces. En lo que sigue suponemos que tanto $c_\textrm{i}$ y $c_\textrm{o}$ son divisibles por $g$.

El único desafío en este diseño es que no se intercambia información entre los grupos $g$. El bloque ResNeXt de
[Referencia fig_resnext_block](https://d2l.ai/chapter_convolutional-modern/resnet.html#fig-resnext-block) modifica el bloque de dos maneras: sitúa la convolución agrupada de $3 \times 3$ entre dos convoluciones de $1 \times 1$. La segunda también restaura el número de canales. La ventaja es
[Referencia subsec_residual-blks](https://d2l.ai/chapter_convolutional-modern/resnet.html#subsec-residual-blks), la conexión residual se sustituye —y se generaliza— mediante una convolución de $1 \times 1$.

La figura de la derecha en [Referencia fig_resnext_block](https://d2l.ai/chapter_convolutional-modern/resnet.html#fig-resnext-block) proporciona un resumen mucho más conciso del bloque de red resultante. También jugará un papel importante en el diseño de CNNs modernas genéricas en [Referencia sec_cnn-design](https://d2l.ai/chapter_convolutional-modern/cnn-design.html#sec-cnn-design). Tenga en cuenta que la idea de convoluciones agrupadas se remonta a la implementación de AlexNet [Krizhevsky.Sutskever.Hinton.2012](https://d2l.ai/chapter_references/zreferences.html). Al distribuir la red a través de dos GPUs con memoria limitada, la implementación trató a cada GPU como su propio canal sin efectos nocivos.

La siguiente implementación de la clase `ResNeXtBlock` toma como argumento `groups` ($g$), con `bot_channels` ($b$) canales intermedios (cuello de botella). Por último, cuando necesitamos reducir la altura y el ancho de la representación, añadimos un paso de $2$ al establecer `use_1x1conv=True, strides=2`.


In [ ]:
class ResNeXtBlock(nn.Module):  #@save
    """El bloque ResNeXt."""
    def __init__(self, num_channels, groups, bot_mul, use_1x1conv=False,
                 strides=1):
        super().__init__()
        bot_channels = int(round(num_channels * bot_mul))
        self.conv1 = nn.LazyConv2d(bot_channels, kernel_size=1, stride=1)
        self.conv2 = nn.LazyConv2d(bot_channels, kernel_size=3,
                                   stride=strides, padding=1,
                                   groups=bot_channels//groups)
        self.conv3 = nn.LazyConv2d(num_channels, kernel_size=1, stride=1)
        self.bn1 = nn.LazyBatchNorm2d()
        self.bn2 = nn.LazyBatchNorm2d()
        self.bn3 = nn.LazyBatchNorm2d()
        if use_1x1conv:
            self.conv4 = nn.LazyConv2d(num_channels, kernel_size=1,
                                       stride=strides)
            self.bn4 = nn.LazyBatchNorm2d()
        else:
            self.conv4 = None

    def forward(self, X):
        Y = F.relu(self.bn1(self.conv1(X)))
        Y = F.relu(self.bn2(self.conv2(Y)))
        Y = self.bn3(self.conv3(Y))
        if self.conv4:
            X = self.bn4(self.conv4(X))
        return F.relu(Y + X)

Su uso es completamente análogo al del `ResNetBlock` discutido anteriormente. Por ejemplo, cuando se utiliza (`use_1x1conv=False, strides=1`), la entrada y salida son de la misma forma. Alternativamente, el ajuste `use_1x1conv=True, strides=2` reduce la altura de salida y la anchura.


In [ ]:
blk = ResNeXtBlock(32, 16, 1)
X = torch.randn(4, 32, 96, 96)
blk(X).shape

## Resumen y debate
Las clases de funciones anidadas son deseables ya que nos permiten obtener estrictamente *más potentes* en lugar de también sutilmente *diferentes* clases de funciones al añadir capacidad. Una manera de lograr esto es dejando que capas adicionales simplemente pasen a través de la entrada a la salida. Las conexiones residuales permiten esto. Como consecuencia, esto cambia el sesgo inductivo de funciones simples siendo de la forma $f(\mathbf{x}) = 0$ a funciones simples luciendo como $f(\mathbf{x}) = \mathbf{x}$.

El mapeo residual puede aprender la función de identidad más fácilmente, como empujar parámetros en la capa de peso a cero. Podemos entrenar una red neural *profunda* efectiva al tener bloques residuales. Las entradas pueden avanzar más rápidamente a través de las conexiones residuales a través de capas. Como consecuencia, podemos entrenar así redes mucho más profundas. Por ejemplo, el papel original de ResNet [He.Zhang.Ren.ea.2016](https://d2l.ai/chapter_references/zreferences.html) permite hasta 152 capas. Otro beneficio de las redes residuales es que nos permite añadir capas, inicializadas como función de identidad, *durando* el proceso de entrenamiento. Después de todo, el comportamiento predeterminado de una capa es dejar que los datos pasen sin cambios. Esto puede acelerar el entrenamiento de redes muy grandes en algunos casos.

Antes de las conexiones residuales, se introdujeron caminos de circunvalación con unidades de atraque para entrenar eficazmente las redes de carreteras con más de 100 capas
[srivastava2015highway](https://d2l.ai/chapter_references/zreferences.html).
Usando las funciones de identidad como rutas de bypassing, ResNet funcionó notablemente bien en múltiples tareas de visión computarizada. Las conexiones residuales tuvieron una gran influencia en el diseño de redes neuronales profundas posteriores, de naturaleza convolucional o secuencial. Como introduciremos más adelante, la arquitectura Transformer [Vaswani.Shazeer.Parmar.ea.2017](https://d2l.ai/chapter_references/zreferences.html) adopta conexiones residuales (junto con otras opciones de diseño) y es omnipresente en áreas tan diversas como el lenguaje, la visión, el habla y el aprendizaje de refuerzo.

ResNeXt es un ejemplo de cómo el diseño de las redes neuronales convolucionales ha evolucionado con el tiempo: al ser más frugal con el cálculo y negociarlo con el tamaño de las activaciones (número de canales), permite redes más rápidas y precisas a menor costo. Una forma alternativa de ver convoluciones agrupadas es pensar en una matriz bloque-diagonal para los pesos convolucionales. Tenga en cuenta que hay bastantes tales "trucos" que conducen a redes más eficientes. Por ejemplo, ShiftNet [wu2018shift](https://d2l.ai/chapter_references/zreferences.html) imita los efectos de una convolución $3 \times 3$, simplemente añadiendo activaciones desplazadas a los canales, ofreciendo una mayor complejidad de la función, esta vez sin ningún costo computacional.

Una característica común de los diseños que hemos discutido hasta ahora es que el diseño de la red es bastante manual, basándose principalmente en el ingenio del diseñador para encontrar los hiperparametros de red "correctos". Aunque claramente factible, también es muy costoso en términos de tiempo humano y no hay garantía de que el resultado sea óptimo en ningún sentido. En [Referencia sec_cnn-design](https://d2l.ai/chapter_convolutional-modern/cnn-design.html#sec-cnn-design) discutiremos una serie de estrategias para obtener redes de alta calidad de una manera más automatizada. En particular, revisaremos la noción de *espacios de diseño de red* que llevó a los modelos RegNetX/Y
[Radosavovic.Kosaraju.Girshick.ea.2020](https://d2l.ai/chapter_references/zreferences.html).

## Ejercicios
1. ¿Cuáles son las principales diferencias entre el bloque de Inception en [Referencia fig_inception](https://d2l.ai/chapter_convolutional-modern/googlenet.html#fig-inception) y el bloque residual? ¿Cómo se comparan en términos de cálculo, precisión y las clases de funciones que pueden describir?
1. Consulte la Tabla 1 en el documento de ResNet [He.Zhang.Ren.ea.2016](https://d2l.ai/chapter_references/zreferences.html) para implementar diferentes variantes de la red.
1. Para redes más profundas, ResNet introduce una arquitectura "bottleneck" para reducir la complejidad del modelo.
1. En versiones posteriores de ResNet, los autores cambiaron la estructura de "convolución, normalización por lotes (BatchNorm) y activación" a la estructura de "normalización por lotes (BatchNorm), activación y convolución". Haga esta mejora usted mismo. Vea la Figura 1 en [He.Zhang.Ren.ea.2016*1](https://d2l.ai/chapter_references/zreferences.html) para más detalles.
1. ¿Por qué no podemos simplemente aumentar la complejidad de las funciones sin límite, incluso si las clases de funciones están anidadas?


[Debate del original](https://discuss.d2l.ai/t/86)
